In [14]:
import os
import torch
import torch.nn as nn
import timm
import numpy as np

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [16]:
from PIL import Image
import os

def remove_bad_images(folder):

    bad = []

    for root, dirs, files in os.walk(folder):

        for file in files:

            path = os.path.join(root, file)

            try:
                img = Image.open(path)
                img.verify()

            except:
                bad.append(path)

    print("Bad images:", len(bad))

    for p in bad:
        try:
            os.remove(p)
        except:
            pass

    print("Removed corrupted images")


dataset_path = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\ai_dataset"

remove_bad_images(dataset_path)

Bad images: 0
Removed corrupted images


In [17]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [18]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

In [19]:
train_path = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\ai_dataset\train"
val_path = r"C:\Users\HP\Desktop\desktop\OPEN LAB\OPENLAB_PROJECT\data\ai_dataset\val"

train_dataset = datasets.ImageFolder(train_path, transform=transform)
val_dataset = datasets.ImageFolder(val_path, transform=transform)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

Classes: ['ai', 'real']
Train size: 23242
Val size: 4648


In [20]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [21]:
model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=2
)

model = model.to(device)

In [22]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [23]:
scaler = torch.amp.GradScaler("cuda")

In [24]:
def train_one_epoch(model, loader):

    model.train()
    total_loss = 0

    for images, labels in loader:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)

In [25]:
def validate(model, loader):

    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)

    print(classification_report(all_labels, all_preds))

    return avg_loss

In [26]:
num_epochs = 10

best_val_loss = float("inf")
best_epoch = 0

train_losses = []
val_losses = []

for epoch in range(num_epochs):

    train_loss = train_one_epoch(model, train_loader)

    val_loss = validate(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print("Train Loss:", train_loss)
    print("Val Loss:", val_loss)

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(model.state_dict(), "best_ai_detector.pth")

        print("Best model saved")

print("Training finished")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

              precision    recall  f1-score   support

           0       0.74      0.74      0.74      2324
           1       0.74      0.75      0.74      2324

    accuracy                           0.74      4648
   macro avg       0.74      0.74      0.74      4648
weighted avg       0.74      0.74      0.74      4648

Epoch 1/10
Train Loss: 1.254094849739756
Val Loss: 0.7439076998462416
Best model saved
              precision    recall  f1-score   support

           0       0.88      0.70      0.78      2324
           1       0.75      0.91      0.82      2324

    accuracy                           0.80      4648
   macro avg       0.82      0.80      0.80      4648
weighted avg       0.82      0.80      0.80      4648

Epoch 2/10
Train Loss: 0.5531503958525238
Val Loss: 0.4481043603322277
Best model saved
              precision    recall  f1-score   support

           0       0.91      0.74      0.81      2324
           1       0.78      0.93      0.85      2324

    acc

In [32]:
torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "best_val_loss": best_val_loss
}, "best_ai_detector.pth")

In [33]:
import pandas as pd

log = pd.DataFrame({
    "epoch": list(range(1, num_epochs+1)),
    "train_loss": train_losses,
    "val_loss": val_losses
})

log.to_csv("training_log.csv", index=False)

print("Training log saved")

ValueError: All arrays must be of the same length

In [34]:
checkpoint = torch.load("best_ai_detector.pth")

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"] + 1
best_val_loss = checkpoint["best_val_loss"]

print("Resuming from epoch:", start_epoch)
print("Best validation loss:", best_val_loss)

C:\Users\HP\AppData\Local\Temp\ipykernel_29632\1468684604.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("best_ai_detector.pth")


Resuming from epoch: 17
Best validation loss: 0.34092787295988164


In [31]:
patience = 5
patience_counter = 0

num_epochs = start_epoch + 20   # train 10 more epochs

for epoch in range(start_epoch, num_epochs):

    train_loss = train_one_epoch(model, train_loader)

    val_loss = validate(model, val_loader)

    print(f"Epoch {epoch+1}")
    print("Train Loss:", train_loss)
    print("Val Loss:", val_loss)

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss
        }, "best_ai_detector.pth")

        print("✔ Best model saved")

    else:

        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print("Early stopping triggered")
        break

              precision    recall  f1-score   support

           0       0.94      0.77      0.85      2324
           1       0.80      0.95      0.87      2324

    accuracy                           0.86      4648
   macro avg       0.87      0.86      0.86      4648
weighted avg       0.87      0.86      0.86      4648

Epoch 11
Train Loss: 0.302950427795832
Val Loss: 0.3633481128881239
No improvement (1/5)
              precision    recall  f1-score   support

           0       0.95      0.76      0.85      2324
           1       0.80      0.96      0.88      2324

    accuracy                           0.86      4648
   macro avg       0.88      0.86      0.86      4648
weighted avg       0.88      0.86      0.86      4648

Epoch 12
Train Loss: 0.2868426582151717
Val Loss: 0.34092787295988164
✔ Best model saved
              precision    recall  f1-score   support

           0       0.93      0.77      0.85      2324
           1       0.81      0.94      0.87      2324

    